In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from pyprojroot import here

# 1. Setup the project path and output directory
output_dir = here() / "output" / "figures"
os.makedirs(output_dir, exist_ok=True)

# Data
max_score = 18
average_score = 12.3
high_count = 22
very_high_count = 8
total_studies = high_count + very_high_count

# Percentage calculations
avg_pct = (average_score / max_score) * 100
high_pct = (high_count / total_studies) * 100
vhigh_pct = (very_high_count / total_studies) * 100

# Settings for A0 Poster Quality
# 300 DPI is standard for high-quality print
SAVE_SETTINGS = {
    'dpi': 300,
    'transparent': True,
    'bbox_inches': 'tight'
}

# --- Radial Bar Chart ---
# We set facecolor='none' to ensure the figure background is transparent
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'}, facecolor='none')

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_ylim(0, 100)

radii = [80, 60, 40]
labels = ['Avg Score', 'High Quality', 'Very High Quality']
values = [avg_pct, high_pct, vhigh_pct]
colors = ['#3498db', '#f1c40f', '#2ecc71']

for i, (val, color, label, radius) in enumerate(zip(values, colors, labels, radii)):
    ax.barh(radius, (val / 100) * 2 * np.pi, height=15, color=color, alpha=0.8, label=f"{label} ({val:.1f}%)")
    ax.barh(radius, 2 * np.pi, height=15, color='lightgrey', alpha=0.2, zorder=0)

ax.set_yticklabels([])
ax.set_xticklabels([])
ax.spines['polar'].set_visible(False)
ax.grid(False)

# Optional: Set text color to white or a high-contrast color if your poster background is dark
text_color = 'black' 

for i, (label, radius, val) in enumerate(zip(labels, radii, values)):
    text = f"{average_score}/{max_score}" if label == 'Avg Score' else f"{[high_count, very_high_count][i-1]} studies"
    ax.text(0, radius, f" {label}: {text}", va='center', fontweight='bold', color=text_color)

plt.title("Quality Assessment Summary", pad=20, fontsize=14, fontweight='bold', color=text_color)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)

plt.tight_layout()
plt.savefig(output_dir / 'stacked_gauge_chart.png', **SAVE_SETTINGS)
plt.close()

# --- Gauge Function ---
def draw_gauge(score, max_val, title, filename):
    fig, ax = plt.subplots(figsize=(8, 5), subplot_kw={'projection': 'polar'}, facecolor='none')
    
    boundaries = [0, 10, 12, 15, 18]
    cat_colors = ['#e74c3c', '#f39c12', '#f1c40f', '#2ecc71']
    
    for i in range(len(boundaries)-1):
        start = (boundaries[i] / max_val) * np.pi
        end = (boundaries[i+1] / max_val) * np.pi
        ax.bar(x=start + (end-start)/2, height=1, width=end-start, bottom=2, 
               color=cat_colors[i], alpha=0.6, align='center')
        
    needle_pos = (score / max_val) * np.pi
    ax.annotate('', xy=(needle_pos, 2.9), xytext=(0, 0),
                arrowprops=dict(arrowstyle="wedge,tail_width=0.5", color='black', lw=2))
    
    ax.set_thetamin(0)
    ax.set_thetamax(180)
    ax.set_theta_offset(np.pi)
    ax.set_theta_direction(-1)
    
    ticks = [0, 5, 10, 12, 15, 18]
    ax.set_xticks([(t/max_val)*np.pi for t in ticks])
    ax.set_xticklabels([str(t) for t in ticks], color=text_color)
    
    ax.set_yticklabels([])
    ax.grid(False)
    ax.spines['polar'].set_visible(False)
    
    plt.title(f"{title}\nAverage Score: {score}/{max_val}", pad=20, fontsize=14, fontweight='bold', color=text_color)
    ax.text(np.pi/2, 0.5, f"Categorized as\n{'High' if score >= 12.3 else '?'}", 
            ha='center', va='center', fontsize=12, fontweight='bold', color=text_color)

    plt.tight_layout()
    plt.savefig(output_dir / filename, **SAVE_SETTINGS)
    plt.close()

draw_gauge(average_score, max_score, "Quality Assessment: EPIFORGE Average", "gauge_average.png")